# Beba Salama — High-Risk Crash Classifier
### Model training, the data leakage fix, evaluation, and SHAP explainability

This notebook documents the actual investigative process behind the trained
LightGBM model referenced in the project README and dashboard — including a
data leakage problem that was found and fixed before any "final" numbers
were produced.

**Data:** World Bank / Ma3Route Kenya Road Traffic Crashes (2012–2023), 31,064 records.
**Target:** `high_risk` — a composite severity flag engineered in `src/features.py`.


In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, classification_report, confusion_matrix,
                              precision_recall_curve, precision_score, recall_score, f1_score)
import lightgbm as lgb
import shap

from data_prep import load_layer1
from features import engineer_features, TARGET

df = load_layer1('../data/raw/ma3route_crashes_algorithmcode.csv')
df = engineer_features(df)
print(f"Loaded {len(df):,} crash records")
df[['hour','dow','month','is_night','is_rush_hour','is_weekend',
    'latitude','longitude','n_crash_reports','contains_matatu_words',
    'contains_motorcycle_words','severity','high_risk']].head()


Loaded 31,064 crash records


,hour,dow,month,is_night,is_rush_hour,is_weekend,latitude,longitude,n_crash_reports,contains_matatu_words,contains_motorcycle_words,severity,high_risk
0,20,2,6,0,0,0,-1.263030,36.764374,1,0,0,2,0
1,6,4,8,0,1,0,-0.829710,37.037820,1,0,0,7,1
2,17,4,5,0,1,0,-1.125301,37.003297,1,0,0,2,0
3,18,4,5,0,1,0,-1.740958,37.129026,1,0,0,2,0
4,21,4,5,1,0,0,-1.259392,36.842321,1,0,0,7,1


## 1. The data leakage problem

Before training anything, it's worth looking hard at how the target was built:

```
severity = n_crash_reports*2 + fatality*5 + pedestrian*3 + matatu*2 + motorcycle*2
high_risk = 1 if severity >= 5 else 0
```

`n_crash_reports`, `contains_matatu_words`, and `contains_motorcycle_words` were
originally on the candidate feature list. But they're **also inputs to the formula
that defines the label**. That's leakage — the model wouldn't be predicting risk,
it would be partially re-deriving its own answer key.

Proof, not assertion:


In [2]:
# Every crash with 3+ independent reports and ZERO other severity flags
# is *automatically* high_risk, purely by construction of the formula.
zero_flags = df[(df['contains_fatality_words']==0) & (df['contains_pedestrian_words']==0) &
                 (df['contains_matatu_words']==0) & (df['contains_motorcycle_words']==0)]

qualifying = zero_flags[zero_flags['n_crash_reports'] >= 3]
print(f"Crashes with n_crash_reports>=3 and no other flags: {len(qualifying)}")
print(f"Of those, classified high_risk: {qualifying['high_risk'].sum()} / {len(qualifying)}")

print()
print("Correlation of each candidate 'feature' with the target it partly defines:")
for col in ['n_crash_reports','contains_matatu_words','contains_motorcycle_words']:
    print(f"  {col:28s} r = {df[col].corr(df['high_risk']):.3f}")


Crashes with n_crash_reports>=3 and no other flags: 1456
Of those, classified high_risk: 1456 / 1456

Correlation of each candidate 'feature' with the target it partly defines:
  n_crash_reports              r = 0.444
  contains_matatu_words        r = 0.179
  contains_motorcycle_words    r = 0.111


**1,456 out of 1,456.** Every single one. That correlation on `n_crash_reports`
(0.44) isn't a real predictive signal — it's the label leaking into the feature set.

**Decision:** drop `n_crash_reports`, `contains_matatu_words`, `contains_motorcycle_words`
(and, for the same reason, `contains_fatality_words` / `contains_pedestrian_words`,
which were never on the candidate list but carry the same problem) from the model
inputs entirely. Train only on what's genuinely known **before** a crash happens:
time and location.


In [3]:
SAFE_FEATURES = ['hour','dow','month','is_night','is_rush_hour',
                  'is_weekend','latitude','longitude']

X = df[SAFE_FEATURES]
y = df[TARGET]

print(f"Feature set: {SAFE_FEATURES}")
print(f"Positive class rate: {y.mean():.3f}  ({y.sum():,} / {len(y):,} high-risk)")


Feature set: ['hour', 'dow', 'month', 'is_night', 'is_rush_hour', 'is_weekend', 'latitude', 'longitude']
Positive class rate: 0.168  (5,206 / 31,064 high-risk)


## 2. Leaky vs. clean — side by side

Quick demonstration of why this matters in practice, not just in theory.


In [4]:
# LEAKY version (for comparison only — not what we ship)
leaky_features = SAFE_FEATURES + ['n_crash_reports','contains_matatu_words','contains_motorcycle_words']
X_leak = df[leaky_features]
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leak, y, test_size=0.2, random_state=42, stratify=y)
model_leak = lgb.LGBMClassifier(random_state=42, verbose=-1)
model_leak.fit(Xtr_l, ytr_l)
auc_leak = roc_auc_score(yte_l, model_leak.predict_proba(Xte_l)[:,1])

print(f"Leaky model AUC-ROC:  {auc_leak:.4f}   <- inflated, would not survive scrutiny")


Leaky model AUC-ROC:  0.8145   <- inflated, would not survive scrutiny


## 3. Training the honest model

Class imbalance (~16.8% positive) is handled with `scale_pos_weight` rather than
resampling, to keep the training distribution representative of the real
population rate.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

model = lgb.LGBMClassifier(
    random_state=42, verbose=-1,
    scale_pos_weight=scale_pos_weight,
    n_estimators=300, learning_rate=0.05,
    max_depth=6, num_leaves=31, min_child_samples=20,
)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
print(f"\nClean model AUC-ROC: {auc:.4f}")


Train: 24,851  |  Test: 6,213
scale_pos_weight: 4.97



Clean model AUC-ROC: 0.6132


## 4. Threshold tuning

For a safety tool, missing a genuinely high-risk crash (a false negative) is worse
than a false alarm. The decision threshold is tuned on F1 as a starting balance,
but in a production deployment this would be pushed further toward recall.


In [6]:
precisions, recalls, thresholds = precision_recall_curve(y_test, proba)
f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold = thresholds[np.argmax(f1s[:-1])]
preds = (proba >= best_threshold).astype(int)

print(f"Best threshold (max F1): {best_threshold:.3f}\n")
print(classification_report(y_test, preds, target_names=['Lower Risk','High Risk']))

cm = confusion_matrix(y_test, preds)
print(f"Confusion matrix:\n  TN={cm[0,0]}  FP={cm[0,1]}\n  FN={cm[1,0]}  TP={cm[1,1]}")


Best threshold (max F1): 0.485

              precision    recall  f1-score   support

  Lower Risk       0.87      0.66      0.75      5172
   High Risk       0.23      0.52      0.32      1041

    accuracy                           0.63      6213
   macro avg       0.55      0.59      0.53      6213
weighted avg       0.76      0.63      0.68      6213

Confusion matrix:
  TN=3397  FP=1775
  FN=503  TP=538


## 5. Feature importance


In [7]:
importance = pd.DataFrame({
    'feature': SAFE_FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
importance


,feature,importance
6,latitude,2614
7,longitude,2563
0,hour,1431
2,month,1140
1,dow,857
4,is_rush_hour,197
3,is_night,64
5,is_weekend,8


## 6. SHAP — explaining individual predictions

This is the explainability layer the project's Methodology section promises.
Global importance below, then a worked example for one specific high-risk
prediction, showing exactly which factors pushed it there.


In [8]:
explainer = shap.TreeExplainer(model)
sample = X_test.sample(500, random_state=42)
shap_values = explainer.shap_values(sample)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_importance = pd.DataFrame({
    'feature': SAFE_FEATURES,
    'mean_abs_shap': np.abs(sv).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)
shap_importance


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,mean_abs_shap
7,longitude,0.199306
6,latitude,0.190483
0,hour,0.130077
1,dow,0.074994
2,month,0.064824
4,is_rush_hour,0.022374
3,is_night,0.009523
5,is_weekend,0.000787


In [9]:
# Worked example — one specific high-risk prediction, explained
high_risk_mask = model.predict_proba(X_test)[:,1] > 0.5
example_idx = X_test[high_risk_mask].index[0]
example = X_test.loc[[example_idx]]
example_shap = explainer.shap_values(example)
example_sv = example_shap[1] if isinstance(example_shap, list) else example_shap

print(f"Predicted probability: {model.predict_proba(example)[0,1]:.3f}\n")
print("Contribution of each factor to this specific prediction:")
for feat, val in zip(SAFE_FEATURES, example_sv[0]):
    direction = 'increases' if val > 0 else 'decreases'
    print(f"  {feat:15s} {val:+.4f}  ({direction} risk)")


Predicted probability: 0.679

Contribution of each factor to this specific prediction:
  hour            +0.3664  (increases risk)
  dow             -0.0120  (decreases risk)
  month           +0.0092  (increases risk)
  is_night        +0.0065  (increases risk)
  is_rush_hour    +0.0337  (increases risk)
  is_weekend      -0.0002  (decreases risk)
  latitude        +0.3606  (increases risk)
  longitude       +0.1974  (increases risk)


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## 7. Honest summary

- **AUC-ROC: ~0.61** on genuinely held-out data, using only time and location —
  modest, but real. This is not a leaderboard-chasing number; it's a defensible one.
- **Recall on the high-risk class: ~0.52** — the model catches roughly half of
  true high-risk crashes at the tuned threshold. For a resource-allocation and
  early-warning tool, that is a meaningful improvement over no signal at all,
  not a finished product.
- **Location dominates** the model (latitude/longitude carry the most weight),
  followed by hour of day. Day-of-week and weekend flags contribute very little
  once location and hour are already known.
- **What this model does NOT do:** it does not use any information about the
  crash itself (matatu involvement, report volume, fatality wording) — only
  what's knowable in advance of a journey. That's a deliberate constraint, not
  an oversight, and it's the reason the AUC is 0.61 and not the inflated 0.81
  the leaky version produced.

This model and these numbers are what the dashboard's Methodology section
should reference going forward — not placeholder language about a model
"in development."
